# Demonstrating RAG Vector DB Benchmark

This notebook demonstrates how to load the experiment configuration, initialize the pipeline, and run a sample evaluation.

In [ ]:
from src.chunkers.fixed_size_chunker import FixedSizeChunker
from src.core.types import Document
from src.embedders.sentence_transformers_embedder import SentenceTransformersEmbedder
from src.generators.universal_generator import UniversalGenerator
from src.pipeline.rag_pipeline import RAGPipeline
from src.retrievers.factory import build_retriever_from_yaml
from src.utils.config_loader import build_component_configs

# 1. Load config
raw_config = {
    "chunker": {"chunk_size": 256, "overlap": 20},
    "embedder": {"model_name": "all-MiniLM-L6-v2", "device": "cpu", "batch_size": 32},
    "generator": {"model_name": "ollama/llama3", "temperature": 0.0, "max_tokens": 512, "api_base": "http://localhost:11434"},
    "retriever": {"collection_name": "demo_collection", "distance_metric": "cosine", "type": "chroma"},
    "pipeline": {"top_k": 2, "evaluate_retrieval": False, "evaluate_generation": False}
}
exp_config = build_component_configs(raw_config)

# 2. Initialize Components
chunker = FixedSizeChunker(exp_config.chunker)
embedder = SentenceTransformersEmbedder(exp_config.embedder)
generator = UniversalGenerator(exp_config.generator)
retriever = build_retriever_from_yaml(raw_config, embedder)

pipeline = RAGPipeline(retriever=retriever, generator=generator, config=exp_config.pipeline)
print("Pipeline Ready!")

In [ ]:
# 3. Add Sample Documents
doc = Document(id="doc_1", content="The capital of France is Paris. The Eiffel Tower is located there.")
chunks = chunker.chunk(doc)
embeddings = embedder.embed_chunks(chunks)
retriever.add_chunks(chunks, embeddings)
print(f"Added {len(chunks)} chunks to Vector DB.")

In [ ]:
# 4. Ask Question
query = "What is the capital of France?"
result = await pipeline.run(query)

print("Answer:", result.rag_response.response)
print("Cost Latency:", result.total_latency_seconds)
